In [1]:
# -----------------------------
# Imports
# -----------------------------
from pathlib import Path
import re
import ast
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# Display settings for easier inspection
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", None)

# -----------------------------
# Paths
# -----------------------------
REPO_ROOT = Path("..").resolve()
DATA_PROCESSED = REPO_ROOT / "data" / "processed"

DEMO_SUBSET_PARQUET = DATA_PROCESSED / "demo_jobs_subset.parquet"
DEMO_CLEAN_PARQUET = DATA_PROCESSED / "demo_jobs_clean.parquet"
DEMO_JOB_EMB_NPY = DATA_PROCESSED / "demo_job_emb.npy"

print("Subset file exists:", DEMO_SUBSET_PARQUET.exists())
print("Subset file path:", DEMO_SUBSET_PARQUET)

Subset file exists: True
Subset file path: C:\Users\COMPUTER CARE\aeej1\JobPlatform\data\processed\demo_jobs_subset.parquet


In [2]:
# -----------------------------
# Load the reduced demo subset created in Notebook 08
# -----------------------------
demo_jobs = pd.read_parquet(DEMO_SUBSET_PARQUET)

print("Shape:", demo_jobs.shape)
print("\nColumns:")
print(demo_jobs.columns.tolist())

display(demo_jobs.head(3))

Shape: (3720, 16)

Columns:
['job_id', 'experience', 'qualifications', 'location', 'country', 'work_type', 'job_posting_date', 'job_title', 'role', 'job_portal', 'job_description', 'benefits', 'skills', 'responsibilities', 'company_name', 'company_profile']


,job_id,experience,qualifications,location,country,work_type,job_posting_date,job_title,role,job_portal,job_description,benefits,skills,responsibilities,company_name,company_profile
0,692765415301241,2 to 11 Years,B.Com,Minsk,Belarus,Contract,2023-05-13,Sales Representative,Sales Representative,LinkedIn,"Account Executives manage and grow relationships with existing clients or customers. They understand client needs, propose solutions, negotiate contracts, and ensure customer satisfaction, often working closely with sales and customer support teams.","{'Tuition Reimbursement, Stock Options or Equity Grants, Parental Leave, Wellness Programs, Childcare Assistance'}",Sales strategy and planning Account management Customer relationship management Solution selling Sales forecasting Contract negotiation Sales metrics and reporting,"Manage client accounts, negotiate contracts, and achieve revenue targets by selling products or services. Provide ongoing support and solutions to clients. Collaborate with cross-functional teams to meet client needs.",VMware,"{""Sector"":""Software"",""Industry"":""Computer Software"",""City"":""Palo Alto"",""State"":""California"",""Zip"":""94304"",""Website"":""www.vmware.com"",""Ticker"":""VMW"",""CEO"":""Raghu Raghuram""}"
1,370087477853353,0 to 12 Years,B.Com,Berlin,Germany,Temporary,2023-05-31,Sales Representative,Sales Representative,Glassdoor,"Account Executives manage and grow relationships with existing clients or customers. They understand client needs, propose solutions, negotiate contracts, and ensure customer satisfaction, often working closely with sales and customer support teams.","{'Employee Assistance Programs (EAP), Tuition Reimbursement, Profit-Sharing, Transportation Benefits, Parental Leave'}",Sales strategy and planning Account management Customer relationship management Solution selling Sales forecasting Contract negotiation Sales metrics and reporting,"Manage client accounts, negotiate contracts, and achieve revenue targets by selling products or services. Provide ongoing support and solutions to clients. Collaborate with cross-functional teams to meet client needs.",Continental AG,"{""Sector"":""Automotive"",""Industry"":""Automotive"",""City"":""Hanover"",""State"":""N/A"",""Zip"":""N/A"",""Website"":""www.continental-corporation.com"",""Ticker"":""CON"",""CEO"":""Nikolai Setzer""}"
2,862414233805803,4 to 9 Years,BA,Saint George's,Grenada,Temporary,2022-06-25,Sales Representative,Sales Representative,Jobs2Careers,"Account Executives manage and grow relationships with existing clients or customers. They understand client needs, propose solutions, negotiate contracts, and ensure customer satisfaction, often working closely with sales and customer support teams.","{'Transportation Benefits, Professional Development, Bonuses and Incentive Programs, Profit-Sharing, Employee Discounts'}",Sales strategy and planning Account management Customer relationship management Solution selling Sales forecasting Contract negotiation Sales metrics and reporting,"Manage client accounts, negotiate contracts, and achieve revenue targets by selling products or services. Provide ongoing support and solutions to clients. Collaborate with cross-functional teams to meet client needs.",IBM (International Business Machines Corporation),"{""Sector"":""Technology/IT Services"",""Industry"":""Technology"",""City"":""Armonk"",""State"":""NY"",""Zip"":""10504"",""Website"":""https://www.ibm.com/"",""Ticker"":""IBM"",""CEO"":""Arvind Krishna""}"


In [3]:
# -----------------------------
# Defensive schema checks
#
# Notebook 09 expects a stable subset schema from Notebook 08.
# If any essential column is missing, we either:
# - create a safe fallback, or
# - stop with a clear error message
# -----------------------------
required_columns = [
    "job_id",
    "experience",
    "qualifications",
    "location",
    "country",
    "work_type",
    "job_posting_date",
    "job_title",
    "job_portal",
    "job_description",
    "benefits",
    "skills",
    "responsibilities",
    "company_name",
    "company_profile",
]

missing_required = [c for c in required_columns if c not in demo_jobs.columns]
if missing_required:
    raise ValueError(f"Missing required columns in demo_jobs_subset.parquet: {missing_required}")

# If role is missing for any reason, fall back to job_title
if "role" not in demo_jobs.columns:
    demo_jobs["role"] = demo_jobs["job_title"]

# Fill blank role values with job_title
demo_jobs["role"] = demo_jobs["role"].replace("", np.nan)
demo_jobs["role"] = demo_jobs["role"].fillna(demo_jobs["job_title"])

print("Columns after defensive checks:")
print(demo_jobs.columns.tolist())

Columns after defensive checks:
['job_id', 'experience', 'qualifications', 'location', 'country', 'work_type', 'job_posting_date', 'job_title', 'role', 'job_portal', 'job_description', 'benefits', 'skills', 'responsibilities', 'company_name', 'company_profile']


In [4]:
# -----------------------------
# Helper: safely clean text
# -----------------------------
def clean_text(x):
    if pd.isna(x):
        return ""
    s = str(x).replace("\u00a0", " ")
    s = re.sub(r"\s+", " ", s)
    return s.strip()


# -----------------------------
# Helper: normalize skill phrase
# -----------------------------
def normalize_skill_phrase(skill):
    s = clean_text(skill).lower()
    s = s.strip(" ,;:.()[]{}<>-/\\|")
    return s


# -----------------------------
# Helper: split long compressed skill blobs
#
# The Kaggle dataset sometimes stores "skills" as one long
# bracketed phrase rather than a clean comma-separated list.
#
# This function tries to split such text into smaller chunks
# using common phrase boundaries.
# -----------------------------
def split_compressed_skill_blob(text):
    text = normalize_skill_phrase(text)

    if not text:
        return []

    # Remove surrounding square brackets if present
    text = text.strip("[]").strip()

    # First, normalize common separators if they exist
    text = text.replace("/", " / ")
    text = re.sub(r"\s+", " ", text).strip()

    # Common multi-word business / technical skill phrases
    # used as "anchors" to help recover meaningful chunks.
    anchor_phrases = [
        "sales strategy and planning",
        "account management",
        "customer relationship management",
        "solution selling",
        "sales forecasting",
        "contract negotiation",
        "sales metrics and reporting",
        "cloud infrastructure",
        "cloud systems engineering",
        "devops practices",
        "disaster recovery",
        "automation",
        "security",
        "aws",
        "azure",
        "gcp",
        "terraform",
        "kubernetes",
        "docker",
        "python",
        "network security",
        "cloud security",
        "data analysis",
        "project management",
        "stakeholder management",
        "user research",
        "frontend development",
        "backend development",
        "seo",
        "content strategy",
        "social media management",
    ]

    found = []

    for phrase in anchor_phrases:
        if phrase in text:
            found.append(phrase)

    # If we found anchored phrases, return those
    if found:
        return sorted(set(found))

    # Fallback:
    # split on commas/semicolons if available
    parts = re.split(r"[;,]", text)
    parts = [normalize_skill_phrase(p) for p in parts if normalize_skill_phrase(p)]

    # Keep only reasonably short chunks
    filtered = []
    for p in parts:
        # Drop overly long chunks that are probably sentence-like
        if len(p.split()) > 8:
            continue
        filtered.append(p)

    if filtered:
        return sorted(set(filtered))

    # Final fallback: return the whole cleaned blob only if not too large
    if len(text.split()) <= 8:
        return [text]

    return []


# -----------------------------
# Helper: parse skills into a list
#
# Handles:
# - list-like strings
# - comma/semicolon separated strings
# - compressed bracketed blobs
# -----------------------------
def parse_skills(x):
    if pd.isna(x):
        return []

    s = str(x).strip()
    if not s:
        return []

    # Try list-like parsing first
    if s.startswith("[") and s.endswith("]"):
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, list):
                cleaned = [normalize_skill_phrase(v) for v in parsed if normalize_skill_phrase(v)]
                return sorted(set(cleaned))
        except Exception:
            # If literal_eval fails, treat it as a compressed blob
            return split_compressed_skill_blob(s)

    # If separators exist, split directly
    if "," in s or ";" in s:
        parts = re.split(r"[;,]", s)
        parts = [normalize_skill_phrase(p) for p in parts if normalize_skill_phrase(p)]
        return sorted(set(parts))

    # Otherwise treat it as a compressed skill blob
    return split_compressed_skill_blob(s)


# -----------------------------
# Helper: build a short snippet
# for dashboard card previews
# -----------------------------
def build_description_snippet(text, max_chars=220):
    text = clean_text(text)
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + "..."

In [5]:
# -----------------------------
# Re-clean the important text columns
# and parse skills into list form
# -----------------------------
TEXT_COLUMNS = [
    "experience",
    "qualifications",
    "location",
    "country",
    "work_type",
    "job_posting_date",
    "job_title",
    "role",
    "job_portal",
    "job_description",
    "benefits",
    "responsibilities",
    "company_name",
    "company_profile",
]

for col in TEXT_COLUMNS:
    demo_jobs[col] = demo_jobs[col].apply(clean_text)

# Parse skills into a normalized list
demo_jobs["job_skills_list"] = demo_jobs["skills"].apply(parse_skills)

display(
    demo_jobs[[
        "job_title",
        "role",
        "company_name",
        "location",
        "skills",
        "job_skills_list"
    ]].head(5)
)

,job_title,role,company_name,location,skills,job_skills_list
0,Sales Representative,Sales Representative,VMware,Minsk,Sales strategy and planning Account management Customer relationship management Solution selling Sales forecasting Contract negotiation Sales metrics and reporting,"[account management, contract negotiation, customer relationship management, sales forecasting, sales metrics and reporting, sales strategy and planning, solution selling]"
1,Sales Representative,Sales Representative,Continental AG,Berlin,Sales strategy and planning Account management Customer relationship management Solution selling Sales forecasting Contract negotiation Sales metrics and reporting,"[account management, contract negotiation, customer relationship management, sales forecasting, sales metrics and reporting, sales strategy and planning, solution selling]"
2,Sales Representative,Sales Representative,IBM (International Business Machines Corporation),Saint George's,Sales strategy and planning Account management Customer relationship management Solution selling Sales forecasting Contract negotiation Sales metrics and reporting,"[account management, contract negotiation, customer relationship management, sales forecasting, sales metrics and reporting, sales strategy and planning, solution selling]"
3,Sales Representative,Sales Representative,UFP Industries,Noumea,Sales strategy and planning Account management Customer relationship management Solution selling Sales forecasting Contract negotiation Sales metrics and reporting,"[account management, contract negotiation, customer relationship management, sales forecasting, sales metrics and reporting, sales strategy and planning, solution selling]"
4,Sales Representative,Sales Representative,Vodafone,Apia,Sales strategy and planning Account management Customer relationship management Solution selling Sales forecasting Contract negotiation Sales metrics and reporting,"[account management, contract negotiation, customer relationship management, sales forecasting, sales metrics and reporting, sales strategy and planning, solution selling]"


In [6]:
# -----------------------------
# Building a rich semantic text representation for each job.
#
# This is the field that will be embedded and used
# by the recommendation engine.
# -----------------------------
def build_demo_job_text(row):
    skills_str = ", ".join(row["job_skills_list"])

    parts = [
        f"Job title: {row['job_title']}.",
        f"Role: {row['role']}.",
        f"Company: {row['company_name']}.",
        f"Location: {row['location']}, {row['country']}.",
        f"Work type: {row['work_type']}.",
        f"Experience required: {row['experience']}.",
        f"Qualifications: {row['qualifications']}.",
        f"Skills: {skills_str}.",
        f"Job description: {row['job_description']}.",
        f"Responsibilities: {row['responsibilities']}.",
        f"Benefits: {row['benefits']}.",
        f"Company profile: {row['company_profile']}.",
        f"Job portal: {row['job_portal']}.",
    ]

    text = " ".join([clean_text(p) for p in parts if clean_text(p)])
    return clean_text(text)

demo_jobs["job_text"] = demo_jobs.apply(build_demo_job_text, axis=1)

# Shorter snippet for frontend card view
demo_jobs["description_snippet"] = demo_jobs["job_description"].apply(build_description_snippet)

display(
    demo_jobs[[
        "job_id",
        "job_title",
        "company_name",
        "location",
        "work_type",
        "job_skills_list",
        "description_snippet",
        "job_text"
    ]].head(3)
)

,job_id,job_title,company_name,location,work_type,job_skills_list,description_snippet,job_text
0,692765415301241,Sales Representative,VMware,Minsk,Contract,"[account management, contract negotiation, customer relationship management, sales forecasting, sales metrics and reporting, sales strategy and planning, solution selling]","Account Executives manage and grow relationships with existing clients or customers. They understand client needs, propose solutions, negotiate contracts, and ensure customer satisfaction, often working closely with sale...","Job title: Sales Representative. Role: Sales Representative. Company: VMware. Location: Minsk, Belarus. Work type: Contract. Experience required: 2 to 11 Years. Qualifications: B.Com. Skills: account management, contract negotiation, customer relationship management, sales forecasting, sales metrics and reporting, sales strategy and planning, solution selling. Job description: Account Executives manage and grow relationships with existing clients or customers. They understand client needs, propose solutions, negotiate contracts, and ensure customer satisfaction, often working closely with sales and customer support teams.. Responsibilities: Manage client accounts, negotiate contracts, and achieve revenue targets by selling products or services. Provide ongoing support and solutions to clients. Collaborate with cross-functional teams to meet client needs.. Benefits: {'Tuition Reimbursement, Stock Options or Equity Grants, Parental Leave, Wellness Programs, Childcare Assistance'}. Company profile: {""Sector"":""Software"",""Industry"":""Computer Software"",""City"":""Palo Alto"",""State"":""California"",""Zip"":""94304"",""Website"":""www.vmware.com"",""Ticker"":""VMW"",""CEO"":""Raghu Raghuram""}. Job portal: LinkedIn."
1,370087477853353,Sales Representative,Continental AG,Berlin,Temporary,"[account management, contract negotiation, customer relationship management, sales forecasting, sales metrics and reporting, sales strategy and planning, solution selling]","Account Executives manage and grow relationships with existing clients or customers. They understand client needs, propose solutions, negotiate contracts, and ensure customer satisfaction, often working closely with sale...","Job title: Sales Representative. Role: Sales Representative. Company: Continental AG. Location: Berlin, Germany. Work type: Temporary. Experience required: 0 to 12 Years. Qualifications: B.Com. Skills: account management, contract negotiation, customer relationship management, sales forecasting, sales metrics and reporting, sales strategy and planning, solution selling. Job description: Account Executives manage and grow relationships with existing clients or customers. They understand client needs, propose solutions, negotiate contracts, and ensure customer satisfaction, often working closely with sales and customer support teams.. Responsibilities: Manage client accounts, negotiate contracts, and achieve revenue targets by selling products or services. Provide ongoing support and solutions to clients. Collaborate with cross-functional teams to meet client needs.. Benefits: {'Employee Assistance Programs (EAP), Tuition Reimbursement, Profit-Sharing, Transportation Benefits, Parental Leave'}. Company profile: {""Sector"":""Automotive"",""Industry"":""Automotive"",""City"":""Hanover"",""State"":""N/A"",""Zip"":""N/A"",""Website"":""www.continental-corporation.com"",""Ticker"":""CON"",""CEO"":""Nikolai Setzer""}. Job portal: Glassdoor."
2,862414233805803,Sales Representative,IBM (International Business Machines Corporation),Saint George's,Temporary,"[account management, contract negotiation, customer relationship management, sales forecasting, sales metrics and reporting, sales strategy and planning, solution selling]","Account Executives manage and grow relationships with existing clients or customers. They understand client needs, propose solutions, negotiate contracts, and ensure customer satisfaction,

In [7]:
# -----------------------------
# Keeping only the final processed fields needed by:
# - FastAPI
# - Spring Boot
# - React dashboard / job details pages
# -----------------------------
FINAL_COLUMNS = [
    "job_id",
    "job_title",
    "role",
    "company_name",
    "location",
    "country",
    "work_type",
    "job_posting_date",
    "job_portal",
    "experience",
    "qualifications",
    "job_description",
    "description_snippet",
    "benefits",
    "responsibilities",
    "company_profile",
    "job_skills_list",
    "job_text",
]

demo_jobs_clean = demo_jobs[FINAL_COLUMNS].copy()

print("Processed demo jobs shape:", demo_jobs_clean.shape)
print("Processed columns:")
print(demo_jobs_clean.columns.tolist())

display(demo_jobs_clean.head(3))

Processed demo jobs shape: (3720, 18)
Processed columns:
['job_id', 'job_title', 'role', 'company_name', 'location', 'country', 'work_type', 'job_posting_date', 'job_portal', 'experience', 'qualifications', 'job_description', 'description_snippet', 'benefits', 'responsibilities', 'company_profile', 'job_skills_list', 'job_text']


,job_id,job_title,role,company_name,location,country,work_type,job_posting_date,job_portal,experience,qualifications,job_description,description_snippet,benefits,responsibilities,company_profile,job_skills_list,job_text
0,692765415301241,Sales Representative,Sales Representative,VMware,Minsk,Belarus,Contract,2023-05-13,LinkedIn,2 to 11 Years,B.Com,"Account Executives manage and grow relationships with existing clients or customers. They understand client needs, propose solutions, negotiate contracts, and ensure customer satisfaction, often working closely with sales and customer support teams.","Account Executives manage and grow relationships with existing clients or customers. They understand client needs, propose solutions, negotiate contracts, and ensure customer satisfaction, often working closely with sale...","{'Tuition Reimbursement, Stock Options or Equity Grants, Parental Leave, Wellness Programs, Childcare Assistance'}","Manage client accounts, negotiate contracts, and achieve revenue targets by selling products or services. Provide ongoing support and solutions to clients. Collaborate with cross-functional teams to meet client needs.","{""Sector"":""Software"",""Industry"":""Computer Software"",""City"":""Palo Alto"",""State"":""California"",""Zip"":""94304"",""Website"":""www.vmware.com"",""Ticker"":""VMW"",""CEO"":""Raghu Raghuram""}","[account management, contract negotiation, customer relationship management, sales forecasting, sales metrics and reporting, sales strategy and planning, solution selling]","Job title: Sales Representative. Role: Sales Representative. Company: VMware. Location: Minsk, Belarus. Work type: Contract. Experience required: 2 to 11 Years. Qualifications: B.Com. Skills: account management, contract negotiation, customer relationship management, sales forecasting, sales metrics and reporting, sales strategy and planning, solution selling. Job description: Account Executives manage and grow relationships with existing clients or customers. They understand client needs, propose solutions, negotiate contracts, and ensure customer satisfaction, often working closely with sales and customer support teams.. Responsibilities: Manage client accounts, negotiate contracts, and achieve revenue targets by selling products or services. Provide ongoing support and solutions to clients. Collaborate with cross-functional teams to meet client needs.. Benefits: {'Tuition Reimbursement, Stock Options or Equity Grants, Parental Leave, Wellness Programs, Childcare Assistance'}. Company profile: {""Sector"":""Software"",""Industry"":""Computer Software"",""City"":""Palo Alto"",""State"":""California"",""Zip"":""94304"",""Website"":""www.vmware.com"",""Ticker"":""VMW"",""CEO"":""Raghu Raghuram""}. Job portal: LinkedIn."
1,370087477853353,Sales Representative,Sales Representative,Continental AG,Berlin,Germany,Temporary,2023-05-31,Glassdoor,0 to 12 Years,B.Com,"Account Executives manage and grow relationships with existing clients or customers. They understand client needs, propose solutions, negotiate contracts, and ensure customer satisfaction, often working closely with sales and customer support teams.","Account Executives manage and grow relationships with existing clients or customers. They understand client needs, propose solutions, negotiate contracts, and ensure customer satisfaction, often working closely with sale...","{'Employee Assistance Programs (EAP), Tuition Reimbursement, Profit-Sharing, Transportation Benefits, Parental Leave'}","Manage client accounts, negotiate contracts, and achieve revenue targets by selling products or services. Provide ongoing support and solutions to clients. Collaborate with cross-functional teams to meet client needs.","{""Sector"":""Automotive"",""Industry"":""Automotive"",""City"":""Hanover"",""State"":""N/A"",""Zip"":""N/A"",""Website"":""www.continental-corporation.com"",""Ticker"":""CON"",""CEO"":""Nikolai Setzer""}","[account management, contract negotiation, cust

In [8]:
# -----------------------------
# Save the processed demo jobs corpus
# -----------------------------
demo_jobs_clean.to_parquet(DEMO_CLEAN_PARQUET, index=False)

print("Saved processed demo jobs to:", DEMO_CLEAN_PARQUET)

Saved processed demo jobs to: C:\Users\COMPUTER CARE\aeej1\JobPlatform\data\processed\demo_jobs_clean.parquet


In [9]:
# -----------------------------
# Load the same embedding model used elsewhere in the project
# -----------------------------
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embed_model = SentenceTransformer(MODEL_NAME)

print("Model loaded:", MODEL_NAME)

C:\Users\COMPUTER CARE\anaconda3\envs\jobplatform\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Model loaded: sentence-transformers/all-MiniLM-L6-v2


In [10]:
# -----------------------------
# Generating embeddings for the processed demo jobs corpus
# -----------------------------
demo_job_emb = embed_model.encode(
    demo_jobs_clean["job_text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

demo_job_emb = np.asarray(demo_job_emb, dtype=np.float32)

print("Embeddings shape:", demo_job_emb.shape)

Batches:   0%|          | 0/117 [00:00<?, ?it/s]

Embeddings shape: (3720, 384)


In [11]:
# -----------------------------
# Saving demo job embeddings for FastAPI loading
# -----------------------------
np.save(DEMO_JOB_EMB_NPY, demo_job_emb)

print("Saved demo job embeddings to:", DEMO_JOB_EMB_NPY)

Saved demo job embeddings to: C:\Users\COMPUTER CARE\aeej1\JobPlatform\data\processed\demo_job_emb.npy


In [12]:
# -----------------------------
# Inspecting a few processed rows to make sure the output looks good
# -----------------------------
display(
    demo_jobs_clean[[
        "job_title",
        "role",
        "company_name",
        "location",
        "work_type",
        "experience",
        "job_portal",
        "job_skills_list",
        "description_snippet"
    ]].head(50)
)

,job_title,role,company_name,location,work_type,experience,job_portal,job_skills_list,description_snippet
0,Sales Representative,Sales Representative,VMware,Minsk,Contract,2 to 11 Years,LinkedIn,"[account management, contract negotiation, customer relationship management, sales forecasting, sales metrics and reporting, sales strategy and planning, solution selling]","Account Executives manage and grow relationships with existing clients or customers. They understand client needs, propose solutions, negotiate contracts, and ensure customer satisfaction, often working closely with sale..."
1,Sales Representative,Sales Representative,Continental AG,Berlin,Temporary,0 to 12 Years,Glassdoor,"[account management, contract negotiation, customer relationship management, sales forecasting, sales metrics and reporting, sales strategy and planning, solution selling]","Account Executives manage and grow relationships with existing clients or customers. They understand client needs, propose solutions, negotiate contracts, and ensure customer satisfaction, often working closely with sale..."
2,Sales Representative,Sales Representative,IBM (International Business Machines Corporation),Saint George's,Temporary,4 to 9 Years,Jobs2Careers,"[account management, contract negotiation, customer relationship management, sales forecasting, sales metrics and reporting, sales strategy and planning, solution selling]","Account Executives manage and grow relationships with existing clients or customers. They understand client needs, propose solutions, negotiate contracts, and ensure customer satisfaction, often working closely with sale..."
3,Sales Representative,Sales Representative,UFP Industries,Noumea,Full-Time,3 to 15 Years,CareerBuilder,"[account management, contract negotiation, customer relationship management, sales forecasting, sales metrics and reporting, sales strategy and planning, solution selling]","Account Executives manage and grow relationships with existing clients or customers. They understand client needs, propose solutions, negotiate contracts, and ensure customer satisfaction, often working closely with sale..."
4,Sales Representative,Sales Representative,Vodafone,Apia,Contract,5 to 13 Years,Jobs2Careers,"[account management, contract negotiation, customer relationship management, sales forecasting, sales metrics and reporting, sales strategy and planning, solution selling]","Account Executives manage and grow relationships with existing clients or customers. They understand client needs, propose solutions, negotiate contracts, and ensure customer satisfaction, often working closely with sale..."
5,Sales Representative,Sales Representative,Agricultural Bank of China,Thimphu,Temporary,0 to 9 Years,Idealist,"[account management, contract negotiation, customer relationship management, sales forecasting, sales metrics and reporting, sales strategy and planning, solution selling]","Account Executives manage and grow relationships with existing clients or customers. They understand client needs, propose solutions, negotiate contracts, and ensure customer satisfaction, often working closely with sale..."
6,Sales Representative,Sales Representative,KKR,Minsk,Intern,4 to 8 Years,Glassdoor,"[account management, contract negotiation, customer relationship management, sales forecasting, sales metrics and reporting, sales strategy and planning, solution selling]","Account Executives manage and grow relationships with existing clients or customers. They understand client needs, propose solutions, negotiate contracts, and ensure customer satisfaction, often working closely with sale..."
7,Sales Representative,Sales Representative,Commercial Metals,Managua,Full-Time,1 to 14 Years,LinkedIn,"[account management, contract negotiation, customer relationship management, sales forecasting, sales metrics and reporting, sales strategy and planning, solution selling]","Account Executives manage and grow relationships with existing clients or customers. They

In [13]:
# -----------------------------
# Verifying that the saved files exist and look usable
# -----------------------------
print("Processed parquet exists:", DEMO_CLEAN_PARQUET.exists())
print("Embedding file exists:", DEMO_JOB_EMB_NPY.exists())

# Reloading parquet to confirm schema on disk
check_jobs = pd.read_parquet(DEMO_CLEAN_PARQUET)

print("\nReloaded processed jobs shape:", check_jobs.shape)
print("Reloaded processed columns:")
print(check_jobs.columns.tolist())

Processed parquet exists: True
Embedding file exists: True

Reloaded processed jobs shape: (3720, 18)
Reloaded processed columns:
['job_id', 'job_title', 'role', 'company_name', 'location', 'country', 'work_type', 'job_posting_date', 'job_portal', 'experience', 'qualifications', 'job_description', 'description_snippet', 'benefits', 'responsibilities', 'company_profile', 'job_skills_list', 'job_text']


## Demo Job Corpus Preparation and Embedding Summary

This notebook converts the reduced demo job subset into a final recommendation corpus. The dataset is cleaned, important display fields are retained, job skills are parsed into list form, and a rich semantic `job_text` field is created for each posting.

This processed corpus is then embedded using the same sentence-transformer model as the rest of the project. The resulting files are saved for direct use by the FastAPI recommendation service in the final user-facing demo.